# Event Response Analysis: Prediction Markets vs. Polls

**DS2500 — Programming with Data | Northeastern University**  
**Team:** Sebastian Brookes, Shiven Mishra

How did prediction markets (Polymarket) and traditional polls (FiveThirtyEight) respond to
major campaign events during the 2024 U.S. presidential election? This notebook measures
**reaction speed**, **intensity**, and **directional accuracy** for five key events.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt

# add project root to path so we can import from src/
PROJECT_ROOT = str(Path.cwd().parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.analysis.events.event_response import (
    EVENTS,
    load_data,
    compute_swing_average,
    compute_538_swing_timeseries,
    build_reaction_summary,
)
from src.visualize.event_plots import (
    apply_style,
    plot_event_timeline,
    plot_reaction_scoreboard,
    plot_indexed_event_study,
)

apply_style()

# lower DPI for inline display
plt.rcParams["figure.dpi"] = 150

## Methodology

### Events analyzed

We focus on **five major campaign events** that fell within the March–September 12, 2024
overlap window (the period where both Polymarket and FiveThirtyEight have daily data for
swing states):

| #   | Event                       | Date    | Expected Direction |
| --- | --------------------------- | ------- | ------------------ |
| 1   | Biden-Trump Debate          | June 27 | pro-Trump          |
| 2   | Trump Assassination Attempt | July 13 | pro-Trump          |
| 3   | Biden Drops Out             | July 21 | pro-Harris         |
| 4   | Walz VP Pick                | Aug 6   | neutral            |
| 5   | Harris-Trump Debate         | Sept 10 | pro-Harris         |

### Why swing states?

National polls and probabilities wash out the signal: California will go blue and Alabama
will go red regardless of campaign events. By averaging across the seven swing states
(AZ, GA, MI, NV, NC, PA, WI), we isolate the _marginal_ effect of each event on the
states that actually decide the election.

### Measuring reactions

For each event, we compute:

- **Baseline** — average Trump lead over the 3 days before the event
- **Immediate** — average Trump lead on the event day and the day after
- **Settled** — average Trump lead over days 2–7 after the event
- **Shift** — settled minus baseline (the sustained change)
- **z-score** — shift divided by the source’s daily volatility (standard deviation of
  day-over-day changes), so we can compare across sources on a common scale


In [ ]:
pm, p538 = load_data()

print(f"Polymarket:      {pm.shape[0]:,} rows, {pm.shape[1]} cols")
print(f"FiveThirtyEight: {p538.shape[0]:,} rows, {p538.shape[1]} cols")
print()
print("Polymarket sample:")
display(pm.head())
print("\nFiveThirtyEight sample:")
display(p538.head())

In [ ]:
pm_swing = compute_swing_average(pm)
p538_swing = compute_538_swing_timeseries(p538)

print(f"Polymarket swing avg:      {pm_swing.index.min().date()} to {pm_swing.index.max().date()}  ({len(pm_swing)} days)")
print(f"FiveThirtyEight swing avg: {p538_swing.index.min().date()} to {p538_swing.index.max().date()}  ({len(p538_swing)} days)")

In [ ]:
plot_event_timeline(pm_swing, p538_swing, save=False)
plt.show()

### Timeline interpretation

Both sources follow the same broad narrative arc: Trump builds a lead through June and
early July (debate performance + assassination attempt sympathy), then **Biden dropping
out on July 21 is the single biggest inflection point of the campaign** — both series
reverse sharply as Harris replaces Biden.

Key observations:

- **Polymarket reacted instantly** to the dropout — traders repriced overnight. The
  FiveThirtyEight average took several days to reflect the shift as new polls trickled in.
- Both sources show a modest Harris bump from the September debate, though the signal is
  weaker than the dropout effect.
- The dual-axis alignment keeps the zero line common to both series so that "above zero =
  Trump leads" and "below zero = Harris leads" is visually consistent.


In [ ]:
summary = build_reaction_summary(pm, p538)

display_cols = ["event", "source", "expected", "baseline", "settled",
                "shift", "shift_z", "direction_match", "has_data"]
display(summary[display_cols].style.format({
    "baseline": "{:.4f}",
    "settled": "{:.4f}",
    "shift": "{:+.4f}",
    "shift_z": "{:+.1f}σ",
}).set_caption("Reaction summary: all events × both sources"))

In [ ]:
plot_reaction_scoreboard(summary, save=False)
plt.show()

### Scoreboard interpretation

The scoreboard reveals two important patterns:

1. **Direction accuracy** — Polymarket shifted in the expected direction for 4 of 5 events;
   FiveThirtyEight managed 3 of 5. Both sources showed an unexpected pro-Trump reaction to
   the Walz VP pick (labelled neutral/expected, so any large move counts as overreaction).

2. **Intensity** — Polymarket’s z-scored reactions are consistently larger, meaning that
   markets moved further relative to their own normal daily noise. This is consistent with
   markets processing new information in near-real-time, while poll aggregates smooth and
   dilute the signal over days.

3. **Data gaps** — FiveThirtyEight has a coverage gap around the Harris-Trump debate
   (Sept 10), which falls near the end of our overlap window (Sept 12). This limits our
   ability to measure the settled reaction for that event in the polling data.


In [ ]:
plot_indexed_event_study(pm_swing, p538_swing, save=False)
plt.show()

### Biden dropout interpretation

The indexed event study makes the speed difference vivid:

- **Polymarket** began repricing on Day 0 (July 21), but the steepest single-day drops
  came on **Days 1 and 3** — not Day 0 itself. Day 0 was a Sunday afternoon announcement
  with thin trading volume; Day 1 (Monday) brought a full trading session plus Harris's
  rapid consolidation of delegate endorsements, giving traders concrete information to
  price in. The Day 3 drop coincides with the wave of major endorsements (Pelosi, Schumer,
  and the Congressional Black Caucus all backed Harris within 72 hours), which removed the
  last remaining uncertainty about a contested convention.
- **FiveThirtyEight** took several more days to reflect the shift. The lag annotation on
  the chart shows the gap between each source's steepest single-day drop.

**Why the difference?** Polymarket trades 24/7 — traders can reprice the moment news
breaks. But even markets don't fully reprice on a single day: the dropout itself was only
the first domino, and each subsequent endorsement cascade narrowed the remaining
uncertainty. FiveThirtyEight aggregates polls, and polls take time to field, collect
responses, weight, and publish. This structural latency means poll aggregates always lag
the true shift in public sentiment by at least a few days — and events that unfold over
multiple news cycles (like a contested-then-settled nomination) compound the delay.


## Conclusions

### Summary of findings

1. **Markets respond faster.** Polymarket repriced swing-state probabilities within hours
   of major events, while FiveThirtyEight's poll aggregate lagged by days.

2. **Markets are more often directionally correct.** Polymarket shifted in the expected
   direction for 4 of 5 events; FiveThirtyEight matched expectations for 3 of 5.

3. **Markets produce stronger signals.** Z-scored reaction magnitudes are consistently
   larger for Polymarket, reflecting a higher signal-to-noise ratio relative to each
   source's own daily volatility.

### Limitations

- **Small event sample** — only 5 events, limiting statistical power for direction accuracy.
- **Overlap window ends Sept 12** — we miss the final two months of the campaign where late
  polls converge toward election day.
- **538 data gaps** — irregular poll publication creates coverage gaps, especially near the
  end of our window (Harris-Trump debate).
- **Apples-to-oranges scale** — Polymarket reports win probabilities while 538 reports vote
  shares. Z-scoring normalizes this, but the underlying quantities are fundamentally
  different.

### So what?

The practical takeaway is that markets and polls answer different questions on different
timescales. A newsroom building a live election dashboard should **weight market data for
breaking-event coverage** (assassination attempts, candidate withdrawals, debate nights)
and **switch to poll data for weekly or monthly trend reports** (demographic shifts, turnout
modeling, regional swings). Neither source alone tells the full story — but using both
without understanding their structural lag differences will produce misleading analysis.

For campaigns themselves, the implication is sharper: if your internal tracking relies
solely on poll aggregates, you are seeing the world on a 3–5 day delay during the moments
that matter most. Market prices won't tell you _why_ the race is shifting, but they will
tell you _that_ it is shifting — days before your pollster can confirm it.
